In [2]:
# ============================================================
# PACKAGE 8.2 : CORRECTED LEAKAGE-FREE MISSING VALUE HANDLING
# ============================================================

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import joblib


print("="*70)
print("PACKAGE 8.2 : LEAKAGE-FREE MISSING VALUE HANDLING")
print("="*70)


# ------------------------------------------------------------
# Load Data
# ------------------------------------------------------------

train = pd.read_csv(r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package1_split\Delhi_AQI_Train.csv")
test = pd.read_csv(r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package1_split\Delhi_AQI_Test.csv")


train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])


TARGET = "AQI_Target"
DATE = "Date"


# ------------------------------------------------------------
# Separate
# ------------------------------------------------------------

X_train = train.drop(columns=[TARGET, DATE])
y_train = train[TARGET]


X_test = test.drop(columns=[TARGET, DATE])
y_test = test[TARGET]



# ------------------------------------------------------------
# Identify Column Types
# ------------------------------------------------------------

numeric_cols = X_train.select_dtypes(
    include=["int64","float64"]
).columns


categorical_cols = X_train.select_dtypes(
    include=["object","category","bool"]
).columns



print("\nNumeric Features:")
print(len(numeric_cols))


print("\nCategorical Features:")
print(len(categorical_cols))

print(categorical_cols.tolist())



# ------------------------------------------------------------
# Missing Values BEFORE
# ------------------------------------------------------------

print("\n============================================================")
print("MISSING VALUES BEFORE")
print("============================================================")

print(
    X_train.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)



# ------------------------------------------------------------
# Numeric Imputer
# FIT ONLY ON TRAIN
# ------------------------------------------------------------

numeric_imputer = SimpleImputer(
    strategy="median"
)


X_train_num = numeric_imputer.fit_transform(
    X_train[numeric_cols]
)


X_test_num = numeric_imputer.transform(
    X_test[numeric_cols]
)



# ------------------------------------------------------------
# Categorical Imputer
# FIT ONLY ON TRAIN
# ------------------------------------------------------------

if len(categorical_cols) > 0:

    categorical_imputer = SimpleImputer(
        strategy="most_frequent"
    )

    X_train_cat = categorical_imputer.fit_transform(
        X_train[categorical_cols]
    )

    X_test_cat = categorical_imputer.transform(
        X_test[categorical_cols]
    )

else:

    X_train_cat = np.empty(
        (len(X_train),0)
    )

    X_test_cat = np.empty(
        (len(X_test),0)
    )



# ------------------------------------------------------------
# Reconstruct DataFrames
# ------------------------------------------------------------

X_train_processed = pd.DataFrame(
    X_train_num,
    columns=numeric_cols
)


X_test_processed = pd.DataFrame(
    X_test_num,
    columns=numeric_cols
)



if len(categorical_cols)>0:

    X_train_processed[categorical_cols] = X_train_cat

    X_test_processed[categorical_cols] = X_test_cat



# ------------------------------------------------------------
# Add Date + Target
# ------------------------------------------------------------

train_processed = pd.concat(
    [
        train[[DATE]].reset_index(drop=True),
        X_train_processed.reset_index(drop=True),
        y_train.reset_index(drop=True)
    ],
    axis=1
)


test_processed = pd.concat(
    [
        test[[DATE]].reset_index(drop=True),
        X_test_processed.reset_index(drop=True),
        y_test.reset_index(drop=True)
    ],
    axis=1
)



# ------------------------------------------------------------
# Check Missing Values
# ------------------------------------------------------------

print("\n============================================================")
print("AFTER IMPUTATION")
print("============================================================")


print(
    "Train Missing:",
    train_processed.isnull().sum().sum()
)


print(
    "Test Missing:",
    test_processed.isnull().sum().sum()
)



# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

train_processed.to_csv(
    "Delhi_AQI_Train_Processed.csv",
    index=False
)


test_processed.to_csv(
    "Delhi_AQI_Test_Processed.csv",
    index=False
)


joblib.dump(
    numeric_imputer,
    "Numeric_Median_Imputer.pkl"
)


if len(categorical_cols)>0:
    joblib.dump(
        categorical_imputer,
        "Categorical_Mode_Imputer.pkl"
    )



print("\nSaved Files:")
print("1. Delhi_AQI_Train_Processed.csv")
print("2. Delhi_AQI_Test_Processed.csv")
print("3. Numeric_Median_Imputer.pkl")
print("4. Categorical_Mode_Imputer.pkl")


print("\nPackage 8.2 Completed Successfully")

PACKAGE 8.2 : LEAKAGE-FREE MISSING VALUE HANDLING

Numeric Features:
78

Categorical Features:
2
['City', 'Season']

MISSING VALUES BEFORE
Sentinel_NO2_Lag3    1349
Sentinel_AAI_Lag3    1349
Sentinel_AAI_Lag1    1347
Sentinel_NO2_Lag1    1347
Sentinel_AAI         1346
Sentinel_AAI_MA3     1346
Sentinel_NO2         1346
Sentinel_NO2_MA3     1346
MODIS_AOD_Lag3        935
MODIS_AOD_Lag1        933
MODIS_AOD_MA7         932
MODIS_AOD             932
MODIS_AOD_MA3         932
City                    0
PM10                    0
PM2.5                   0
Month                   0
Year                    0
AQI                     0
Xylene                  0
dtype: int64

AFTER IMPUTATION
Train Missing: 0
Test Missing: 0

Saved Files:
1. Delhi_AQI_Train_Processed.csv
2. Delhi_AQI_Test_Processed.csv
3. Numeric_Median_Imputer.pkl
4. Categorical_Mode_Imputer.pkl

Package 8.2 Completed Successfully
